# Laboratorio / Proyecto Integrador — Pipeline Modular en PySpark

**Proyecto:** Impacto de Eventos Macroeconómicos Globales en la Economía Mexicana  
**Flujo del Pipeline:** `Extract → Validate → Transform & Integrate → Validate → Load → Verify`

## 1. Objetivo y Contexto
Este notebook documenta de principio a fin el proceso seguido para construir, depurar, validar y ejecutar el pipeline modular en PySpark del Proyecto Integrador, además de inspeccionar directamente el archivo Parquet final generado en `data/processed/proyecto_integrado_mensual.parquet`.

### Fuentes integradas (`data/raw/`)
1. **`finanzas_igae_mensual_final.parquet`** (salida del Laboratorio 8): 187 registros mensuales con Finanzas Públicas de la SHCP deflactadas con el INPC (`INGRESOS_REALES`, `GASTOS_REALES`) y las 4 series sectoriales del IGAE (`Total`, `Actividades primarias`, `Actividades secundarias`, `Actividades terciarias`).
2. **`tipo_cambio_fix.csv`** (descarga de Banxico, serie `SF43718`): 8,791 registros diarios desde 1991-11-12 con columnas `fecha` (`dd/mm/aaaa`) y `dato`.
3. **`s&p500_mensual.csv`** (descarga de Yahoo Finance `^GSPC` y `^VIX`): 442 registros mensuales con `fecha` al último día calendario del mes (`ME`), cierre, mínimo, máximo, volatilidad y rendimiento mensual.

### Decisiones técnicas e incidencias resueltas durante la implementación
* **Homologación de granularidad (`1:1` por `PERIODO`):** El tipo de cambio diario se limpió excluyendo `"N/E"` y se agregó por mes. En el S&P 500 se truncó la fecha de fin de mes al primer día del mes (`F.trunc(..., "month")`) para hacerla compatible con la clave `PERIODO` del Laboratorio 8.
* **Estrategia de cruce:** Se utilizó `Inner Join` con `F.broadcast()` sobre las dos series financieras mensuales para que el optimizador Catalyst ejecute `BroadcastHashJoin`, evitando *shuffles* innecesarios.
* **Resolución de entorno en Windows:**
  1. Se definieron `PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` con `sys.executable` antes de importar PySpark para evitar el aviso de `python3` faltante.
  2. Ante la ausencia de `winutils.exe` (`HADOOP_HOME` sin definir al ejecutar `df.write.parquet()`), la etapa `load()` exporta el resultado validado (187 filas) mediante `df.toPandas().to_parquet(..., index=False, engine="pyarrow")`.
  3. Se incorporó limpieza previa de directorio residual (`shutil.rmtree`) en `load()` para evitar `PermissionError [WinError 5]` cuando existía una carpeta previa con el mismo nombre del archivo.


## 2. Preparación del entorno y sesión Spark
Nos aseguramos de que el directorio de trabajo apunte a la **raíz del proyecto** (donde se encuentra `pipeline.py`) para importar correctamente los módulos en `src/`.


In [1]:
import os
import sys
from pathlib import Path

# Configurar el ejecutable de Python antes de iniciar PySpark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Si Jupyter abrió el directorio notebooks/, regresamos a la raíz del proyecto
if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pipeline.py").exists():
    os.chdir("..")

print("Directorio de trabajo:", Path.cwd())
assert Path("pipeline.py").exists(), "Ejecuta este notebook desde la raíz del proyecto o dentro de notebooks/" 


Directorio de trabajo: c:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD


In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProyectoIntegrador_Demostracion")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

jvm = spark._jvm.java.lang.System
print("Spark:", spark.version)
print("Java:", jvm.getProperty("java.version"))
print("Spark UI:", spark.sparkContext.uiWebUrl)


Spark: 3.5.9
Java: 1.8.0_503
Spark UI: http://Fanta:4040


## 3. Etapa `Extract`: Reconocimiento de las tres fuentes
Llamamos directamente a las funciones definidas en `src/ingestion/extract.py` (sin duplicar lógica en el notebook). Para los CSV se usan esquemas explícitos (`StructType`) para proteger el contrato de tipos.


In [3]:
from pipeline import (
    LAB8_PATH,
    TC_PATH,
    SP500_PATH,
    OUTPUT_PATH,
    REPORT_PATH,
    run_pipeline,
)
from src.ingestion.extract import extract_lab8, extract_tipo_cambio, extract_sp500

df_lab8 = extract_lab8(spark, LAB8_PATH)
df_tc_raw = extract_tipo_cambio(spark, TC_PATH)
df_sp_raw = extract_sp500(spark, SP500_PATH)

print("\n--- Esquema Lab 8 ---")
df_lab8.printSchema()
print("Registros Lab 8:", df_lab8.count())
df_lab8.select("PERIODO", "INGRESOS_REALES", "GASTOS_REALES", "Total").show(5, truncate=False)

print("--- Tipo de Cambio (diario) ---")
print("Registros TC diario:", df_tc_raw.count())
df_tc_raw.show(5, truncate=False)

print("--- S&P 500 + VIX (mensual, fecha fin de mes) ---")
print("Registros S&P 500:", df_sp_raw.count())
df_sp_raw.select("fecha", "sp500_cierre", "sp500_ret_m", "vix_cierre").show(5, truncate=False)


[EXTRACT] Leyendo Parquet Lab 8: data/raw/finanzas_igae_mensual_final.parquet
[EXTRACT] Leyendo CSV Tipo de Cambio: data/raw/tipo_cambio_fix.csv
[EXTRACT] Leyendo CSV S&P 500 + VIX: data/raw/s&p500_mensual.csv

--- Esquema Lab 8 ---
root
 |-- PERIODO: date (nullable = true)
 |-- INGRESOS_REALES: double (nullable = true)
 |-- GASTOS_REALES: double (nullable = true)
 |-- Total: double (nullable = true)
 |-- Actividades primarias: double (nullable = true)
 |-- Actividades secundarias: double (nullable = true)
 |-- Actividades terciarias: double (nullable = true)

Registros Lab 8: 187
+----------+--------------------+--------------------+----------------+
|PERIODO   |INGRESOS_REALES     |GASTOS_REALES       |Total           |
+----------+--------------------+--------------------+----------------+
|2016-03-01|5.265423178640631E8 |4.1988668213510334E8|94.1272843004329|
|2019-11-01|4.865511981784747E8 |4.681168725894728E8 |102.895473729605|
|2018-09-01|4.0800993548266786E8|4.8907894043595624E

## 4. Etapas `Validate` (entrada) y `Transform` (homologación mensual)
Validamos las entradas crudas y ejecutamos la mensualización de Banxico y la homologación de `PERIODO` del S&P 500, comprobando que ninguna fuente tenga meses duplicados antes del `join`.


In [4]:
from src.quality.validate import (
    validate_lab8_input,
    validate_tc_input,
    validate_sp500_input,
    validate_prepared_monthly,
    validate_output,
)
from src.transformations.integrate_enrich import (
    prepare_tipo_cambio,
    prepare_sp500,
    integrate_and_enrich,
)

n_lab8 = validate_lab8_input(df_lab8)
n_tc = validate_tc_input(df_tc_raw)
n_sp = validate_sp500_input(df_sp_raw)
print(f"Entradas validadas -> Lab8: {n_lab8} | TC diario: {n_tc} | S&P500: {n_sp}")

df_tc_mensual = prepare_tipo_cambio(df_tc_raw)
df_sp_mensual = prepare_sp500(df_sp_raw)

validate_prepared_monthly(df_tc_mensual, "TC_MENSUAL")
validate_prepared_monthly(df_sp_mensual, "SP500_MENSUAL")

print("Meses únicos TC:", df_tc_mensual.count())
df_tc_mensual.orderBy("PERIODO").show(5, truncate=False)

print("Meses únicos S&P 500:", df_sp_mensual.count())
df_sp_mensual.orderBy("PERIODO").show(5, truncate=False)


[VALIDATE] Revisando entrada Lab 8
[VALIDATE] Revisando entrada Tipo de Cambio
[VALIDATE] Revisando entrada S&P 500
Entradas validadas -> Lab8: 187 | TC diario: 8791 | S&P500: 442
[TRANSFORM] Preparando Tipo de Cambio (USD/MXN - solo cierre mensual)
[TRANSFORM] Homologando PERIODO en S&P 500 (solo cierre)
Meses únicos TC: 421
+----------+---------+
|PERIODO   |TC_CIERRE|
+----------+---------+
|1991-11-01|3.0672   |
|1991-12-01|3.078    |
|1992-01-01|3.0673   |
|1992-02-01|3.0609   |
|1992-03-01|3.062    |
+----------+---------+
only showing top 5 rows

Meses únicos S&P 500: 442
+----------+------------------+
|PERIODO   |SP500_CIERRE      |
+----------+------------------+
|1990-01-01|329.0799865722656 |
|1990-02-01|331.8900146484375 |
|1990-03-01|339.94000244140625|
|1990-04-01|330.79998779296875|
|1990-05-01|361.2300109863281 |
+----------+------------------+
only showing top 5 rows



## 5. Integración `1:1`, Enriquecimiento con Ventanas y Plan Físico (`explain`)
Al unir las tres series mensuales con `Inner Join` y `F.broadcast()`, inspeccionamos el **Plan Físico** para verificar:
1. **`HashAggregate` en dos fases + `Exchange`** en la agrupación mensual del tipo de cambio.
2. **`BroadcastHashJoin`** (`BroadcastExchange`) para cruzar las tablas mensuales sin *shuffles* pesados en el cruce.


In [5]:
df_enriquecido = integrate_and_enrich(df_lab8, df_tc_mensual, df_sp_mensual)

print("=== PLAN FÍSICO (explain formatted) ===")
df_enriquecido.explain("formatted")

print("=== PRIMEROS 8 PERIODOS INTEGRADOS ===")
df_enriquecido.printSchema()
df_enriquecido.show(8, truncate=False)

[TRANSFORM] Integrando Lab 8 + TC_CIERRE + SP500_CIERRE (Inner Join 1:1)
=== PLAN FÍSICO (explain formatted) ===
== Physical Plan ==
* Sort (21)
+- Exchange (20)
   +- * Project (19)
      +- * BroadcastHashJoin Inner BuildRight (18)
         :- * Project (13)
         :  +- * BroadcastHashJoin Inner BuildRight (12)
         :     :- * Filter (3)
         :     :  +- * ColumnarToRow (2)
         :     :     +- Scan parquet  (1)
         :     +- BroadcastExchange (11)
         :        +- * HashAggregate (10)
         :           +- Exchange (9)
         :              +- * HashAggregate (8)
         :                 +- * Project (7)
         :                    +- * Project (6)
         :                       +- * Filter (5)
         :                          +- Scan csv  (4)
         +- BroadcastExchange (17)
            +- * Project (16)
               +- * Filter (15)
                  +- Scan csv  (14)


(1) Scan parquet 
Output [7]: [PERIODO#0, INGRESOS_REALES#1, GASTOS_REALE

## 6. Prueba negativa de calidad (simulación de mes duplicado)
Comprobamos que los contratos de calidad en `src/quality/validate.py` detienen la ejecución si aparece un `PERIODO` duplicado.


In [6]:
# Simulamos en memoria un mes duplicado tomando la primera fila y uniéndola consigo misma
df_con_duplicado = df_lab8.unionByName(df_lab8.limit(1))

try:
    validate_lab8_input(df_con_duplicado)
    raise AssertionError("La validación debió fallar ante un mes duplicado")
except ValueError as err:
    print("Prueba negativa superada. El contrato detuvo el flujo con:")
    print(" ->", err)


[VALIDATE] Revisando entrada Lab 8
Prueba negativa superada. El contrato detuvo el flujo con:
 -> [LAB8] Existen 1 periodos duplicados en Lab 8.


## 7. Ejecución completa del pipeline (`run_pipeline`) y reporte de evidencia
Ejecutamos el orquestador `run_pipeline(spark)` que coordina todas las etapas, guarda el Parquet en `data/processed/proyecto_integrado_mensual.parquet`, verifica su relectura y escribe la evidencia en `reports/ultima_ejecucion.json`.


In [7]:
import json

resultado = run_pipeline(spark)

print("\n=== EVIDENCIA EN reports/ultima_ejecucion.json ===")
reporte = json.loads(Path(REPORT_PATH).read_text(encoding="utf-8"))
print(json.dumps(reporte, indent=2, ensure_ascii=False))


=== INICIO DEL PIPELINE ===
[EXTRACT] Leyendo Parquet Lab 8: data/raw/finanzas_igae_mensual_final.parquet
[EXTRACT] Leyendo CSV Tipo de Cambio: data/raw/tipo_cambio_fix.csv
[EXTRACT] Leyendo CSV S&P 500 + VIX: data/raw/s&p500_mensual.csv
[VALIDATE] Revisando entrada Lab 8
[VALIDATE] Revisando entrada Tipo de Cambio
[VALIDATE] Revisando entrada S&P 500
[VALIDATE] Entradas correctas: {'lab8': 187, 'tipo_cambio_diario': 8791, 'sp500_mensual': 442}
[TRANSFORM] Preparando Tipo de Cambio (USD/MXN - solo cierre mensual)
[TRANSFORM] Homologando PERIODO en S&P 500 (solo cierre)
[TRANSFORM] Integrando Lab 8 + TC_CIERRE + SP500_CIERRE (Inner Join 1:1)
[VALIDATE] Revisando salida integrada
[VALIDATE] Salida correcta: 187 registros
[LOAD] Guardando resultado Parquet en data/processed/proyecto_integrado_mensual.parquet
[VERIFY] Releyendo Parquet generado para confirmar integridad
[VERIFY] Relectura exitosa: 187 filas, 9 columnas
[OK] Resultado publicado en: data/processed/proyecto_integrado_mensual.

## 8. Inspección del archivo Parquet final generado
Leemos directamente `data/processed/proyecto_integrado_mensual.parquet` tanto con **PySpark** como con **Pandas** para verificar su esquema completo (26 columnas), cobertura temporal, ausencia de nulos inesperados y valores finales.


In [8]:
# 8.1 Lectura y auditoría con PySpark
df_final_spark = spark.read.parquet(OUTPUT_PATH)

print("Ruta:", OUTPUT_PATH)
print(f"Dimensiones: {df_final_spark.count()} filas x {len(df_final_spark.columns)} columnas")
print("\nEsquema completo:")
df_final_spark.printSchema()

# Cobertura temporal
df_final_spark.select(
    F.min("PERIODO").alias("PERIODO_INICIAL"),
    F.max("PERIODO").alias("PERIODO_FINAL"),
    F.countDistinct("PERIODO").alias("MESES_UNICOS"),
).show(truncate=False)

# Conteo de nulos por columna (solo el primer mes debe tener 1 nulo en columnas lag/variación)
nulos_resumen = df_final_spark.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_final_spark.columns
])
print("Conteo de nulos por columna:")
for k, v in nulos_resumen.first().asDict().items():
    print(f"  - {k}: {v}")


Ruta: data/processed/proyecto_integrado_mensual.parquet
Dimensiones: 187 filas x 9 columnas

Esquema completo:
root
 |-- PERIODO: date (nullable = true)
 |-- INGRESOS_REALES: double (nullable = true)
 |-- GASTOS_REALES: double (nullable = true)
 |-- Total: double (nullable = true)
 |-- Actividades primarias: double (nullable = true)
 |-- Actividades secundarias: double (nullable = true)
 |-- Actividades terciarias: double (nullable = true)
 |-- TC_CIERRE: double (nullable = true)
 |-- SP500_CIERRE: double (nullable = true)

+---------------+-------------+------------+
|PERIODO_INICIAL|PERIODO_FINAL|MESES_UNICOS|
+---------------+-------------+------------+
|2011-01-01     |2026-07-01   |187         |
+---------------+-------------+------------+

Conteo de nulos por columna:
  - PERIODO: 0
  - INGRESOS_REALES: 0
  - GASTOS_REALES: 0
  - Total: 0
  - Actividades primarias: 0
  - Actividades secundarias: 0
  - Actividades terciarias: 0
  - TC_CIERRE: 0
  - SP500_CIERRE: 0


In [9]:
# 8.2 Visualización tabular completa y resumen estadístico con Pandas
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

df_final_pd = pd.read_parquet(OUTPUT_PATH)
print(f"Forma en Pandas: {df_final_pd.shape}")
print("\nColumnas:", list(df_final_pd.columns))

print("\nPrimeras 5 filas:")
display(df_final_pd.head(5))

print("\nÚltimas 5 filas:")
display(df_final_pd.tail(5))

print("\nResumen estadístico:")
display(df_final_pd.describe().T[["count", "mean", "std", "min", "50%", "max"]])


Forma en Pandas: (187, 9)

Columnas: ['PERIODO', 'INGRESOS_REALES', 'GASTOS_REALES', 'Total', 'Actividades primarias', 'Actividades secundarias', 'Actividades terciarias', 'TC_CIERRE', 'SP500_CIERRE']

Primeras 5 filas:


,PERIODO,INGRESOS_REALES,GASTOS_REALES,Total,Actividades primarias,Actividades secundarias,Actividades terciarias,TC_CIERRE,SP500_CIERRE
0,2011-01-01,4.380374e+08,4.269849e+08,81.059540,79.408239,86.769780,77.907893,12.1519,1286.119995
1,2011-02-01,3.309457e+08,3.598848e+08,80.115615,70.907183,85.142158,77.773358,12.1062,1327.219971
2,2011-03-01,3.418151e+08,3.751086e+08,87.173674,73.788636,92.861741,84.688332,11.9084,1325.829956
3,2011-04-01,3.553345e+08,3.637086e+08,82.957170,74.688351,89.345867,79.788808,11.5278,1363.609985
4,2011-05-01,3.407529e+08,3.688092e+08,86.038632,84.999197,90.934982,83.315218,11.5780,1345.199951



Últimas 5 filas:


,PERIODO,INGRESOS_REALES,GASTOS_REALES,Total,Actividades primarias,Actividades secundarias,Actividades terciarias,TC_CIERRE,SP500_CIERRE
182,2026-03-01,6.287677e+08,7.524762e+08,106.738735,96.176647,102.854397,109.533630,18.0033,6528.520020
183,2026-04-01,5.459729e+08,5.945280e+08,106.613699,104.643588,102.558775,109.026816,17.5118,7209.009766
184,2026-05-01,4.485020e+08,6.737661e+08,108.549177,125.659794,103.810617,110.287602,17.3505,7580.060059
185,2026-06-01,5.347608e+08,7.200194e+08,107.390831,111.311207,102.525170,109.936310,17.4693,7499.359863
186,2026-07-01,4.973950e+08,6.881310e+08,109.210136,101.391452,104.283536,112.444236,17.3288,7489.720215



Resumen estadístico:


,count,mean,std,min,50%,max
INGRESOS_REALES,187.0,4.492032e+08,9.000458e+07,3.019375e+08,4.334781e+08,9.658026e+08
GASTOS_REALES,187.0,5.326184e+08,1.329174e+08,3.487472e+08,4.893167e+08,9.983901e+08
Total,187.0,9.679117e+01,6.634553e+00,7.633145e+01,9.738333e+01,1.092101e+02
Actividades primarias,187.0,9.734705e+01,1.835944e+01,6.012736e+01,9.560924e+01,1.483375e+02
Actividades secundarias,187.0,9.779887e+01,5.058086e+00,6.974816e+01,9.808335e+01,1.073936e+02
Actividades terciarias,187.0,9.618778e+01,8.026706e+00,7.777336e+01,9.665394e+01,1.124442e+02
TC_CIERRE,187.0,1.742122e+01,3.007752e+00,1.152780e+01,1.830750e+01,2.392830e+01
SP500_CIERRE,187.0,3.232483e+03,1.645945e+03,1.131420e+03,2.752060e+03,7.580060e+03


## 9. Cierre de sesión
Antes de ejecutar `spark.stop()`, puedes abrir la **Spark UI** en la URL impresa al inicio para inspeccionar el grupo de trabajo `PROYECTO_PIPELINE`, los *Stages*, el volumen de *Shuffle* y confirmar `Spill = 0 B`.


In [11]:
# Descomenta la siguiente línea cuando termines de inspeccionar Spark UI
spark.stop()
print("SparkSession detenida.")


SparkSession detenida.
